# Fluxo 3 — Ledger Financeiro (Alta Precisão)

**Universidade Mackenzie — MBA Engenharia de Dados**  
Disciplina: Data Collect and Storage | Prof. Filipe Quintieri Lima  
Aluno: Matheus Alves da Silva | RA: 10752559

| Item | Valor |
|---|---|
| Estratégia | Incremental (Insert-Only) |
| Formato de saída | Parquet |
| Destino | MinIO — camada Bronze |
| Precisão | `DecimalType(15,4)` espelha `NUMERIC(15,4)` do PostgreSQL |
| Watermark | Lido do último metadata com `status: success` |

In [1]:
# Importações e variáveis de configuração
import json
import os
from datetime import date, datetime

import boto3
from botocore.client import Config
from botocore.exceptions import ClientError
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.types import (
    DecimalType,
    LongType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

load_dotenv()

PG_HOST     = os.getenv("PG_HOST")
PG_PORT     = int(os.getenv("PG_PORT", "5432"))
PG_DB       = os.getenv("PG_DB")
PG_USER     = os.getenv("PG_USER")
PG_PASSWORD = os.getenv("PG_PASSWORD")
PG_JDBC_URL = f"jdbc:postgresql://{PG_HOST}:{PG_PORT}/{PG_DB}"

MINIO_ENDPOINT   = os.getenv("MINIO_ENDPOINT")
MINIO_ACCESS_KEY = os.getenv("MINIO_ACCESS_KEY")
MINIO_SECRET_KEY = os.getenv("MINIO_SECRET_KEY")
MINIO_BUCKET     = os.getenv("MINIO_BUCKET", "datalake")

METADATA_PREFIX    = "bronze/financeiro/metadata/"
WATERMARK_DEFAULT  = os.getenv("ULTIMA_EXECUCAO", "2000-01-01 00:00:00")

DATA_INGESTAO      = date.today().strftime("%Y-%m-%d")
TIMESTAMP_INGESTAO = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

OUTPUT_PATH = (
    f"s3a://{MINIO_BUCKET}/bronze/financeiro/ledger/"
    f"data_ingestao={DATA_INGESTAO}/transacoes.parquet"
)

print(f"JDBC URL  : {PG_JDBC_URL}")
print(f"MinIO     : {MINIO_ENDPOINT}")
print(f"Output    : {OUTPUT_PATH}")

JDBC URL  : jdbc:postgresql://postgres:5432/postgres
MinIO     : http://minio:9000
Output    : s3a://datalake/bronze/financeiro/ledger/data_ingestao=2026-05-30/transacoes.parquet


In [2]:
# Schema explícito — Alta Precisão Numérica
# DecimalType(15,4) espelha NUMERIC(15,4) do PostgreSQL.
# Impede que o Spark converta silenciosamente para Double/Float,
# causando perda de precisão em valores monetários.
SCHEMA_TRANSACOES = StructType([
    StructField("id",            LongType(),         nullable=False),
    StructField("data_criacao",  TimestampType(),    nullable=False),
    StructField("tipo",          StringType(),       nullable=True),
    StructField("valor",         DecimalType(15, 4), nullable=False),
    StructField("descricao",     StringType(),       nullable=True),
    StructField("conta_origem",  StringType(),       nullable=True),
    StructField("conta_destino", StringType(),       nullable=True),
])

print("Schema definido:")
for f in SCHEMA_TRANSACOES.fields:
    print(f"  {f.name:<16} {f.dataType}")

Schema definido:
  id               LongType()
  data_criacao     TimestampType()
  tipo             StringType()
  valor            DecimalType(15,4)
  descricao        StringType()
  conta_origem     StringType()
  conta_destino    StringType()


In [3]:
# Funções de Metadata e Watermark (boto3 — sem Spark)

def get_minio_client():
    return boto3.client(
        "s3",
        endpoint_url=MINIO_ENDPOINT,
        aws_access_key_id=MINIO_ACCESS_KEY,
        aws_secret_access_key=MINIO_SECRET_KEY,
        config=Config(signature_version="s3v4"),
        region_name="us-east-1",
    )


def ler_watermark(s3_client) -> str:
    """Lista os arquivos de metadata e retorna o timestamp do mais recente com status 'success'."""
    try:
        response = s3_client.list_objects_v2(Bucket=MINIO_BUCKET, Prefix=METADATA_PREFIX)
        objetos = response.get("Contents", [])
        if not objetos:
            print("[INFO] Nenhum metadata encontrado. Usando valor padrão.")
            return WATERMARK_DEFAULT

        for obj in sorted(objetos, key=lambda o: o["LastModified"], reverse=True):
            body = s3_client.get_object(Bucket=MINIO_BUCKET, Key=obj["Key"])
            data = json.loads(body["Body"].read().decode("utf-8"))
            for registro in (data if isinstance(data, list) else [data]):
                if registro.get("status") == "success":
                    print(f"[INFO] Watermark encontrado em '{obj['Key']}': {registro['timestamp']}")
                    return registro["timestamp"]

        print("[WARN] Nenhum metadata com status 'success'. Usando valor padrão.")
    except ClientError as e:
        print(f"[WARN] Erro ao listar metadata: {e}. Usando valor padrão.")
    return WATERMARK_DEFAULT


def upload_metadata(s3_client, registro: dict):
    key = f"{METADATA_PREFIX}{TIMESTAMP_INGESTAO}.json"
    s3_client.put_object(
        Bucket=MINIO_BUCKET,
        Key=key,
        Body=json.dumps(registro).encode("utf-8"),
        ContentType="application/json",
    )
    print(f"[OK] Metadata salvo: s3://{MINIO_BUCKET}/{key}")


print("Funções de metadata definidas.")

Funções de metadata definidas.


In [4]:
# Leitura do watermark antes de inicializar o Spark
s3 = get_minio_client()
ultima_execucao = ler_watermark(s3)

print("=" * 60)
print("FLUXO 3 — Ledger Financeiro (Insert-Only / Alta Precisão)")
print(f"Data de ingestão   : {DATA_INGESTAO}")
print(f"Watermark utilizado: {ultima_execucao}")
print("=" * 60)

[INFO] Nenhum metadata encontrado. Usando valor padrão.
FLUXO 3 — Ledger Financeiro (Insert-Only / Alta Precisão)
Data de ingestão   : 2026-05-30
Watermark utilizado: 2000-01-01 00:00:00


In [5]:
# Inicialização da SparkSession com JARs via Maven
spark = (
    SparkSession.builder
    .appName("Fluxo3_LedgerFinanceiro")
    .config(
        "spark.jars.packages",
        "org.postgresql:postgresql:42.7.2,"
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.262",
    )
    .config("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"SparkSession ativa. Versão: {spark.version}")

SparkSession ativa. Versão: 3.5.0


In [6]:
# Extração — transacoes_financeiras via JDBC
# Mesmo padrão do fluxo2: spark.read.jdbc()
# .schema() é ignorado pelo JDBC — o Spark usa os tipos do ResultSetMetaData.
# O cast explícito abaixo garante DecimalType(15,4) para 'valor'.
query = (
    f"(SELECT * FROM transacoes_financeiras "
    f"WHERE data_criacao > '{ultima_execucao}') AS t"
)

print(f"[INFO] Extraindo transacoes_financeiras com data_criacao > {ultima_execucao}")

df_transacoes = spark.read.jdbc(
    url=PG_JDBC_URL,
    table=query,
    properties={
        "user": PG_USER,
        "password": PG_PASSWORD,
        "driver": "org.postgresql.Driver",
    },
)

# Cast explícito: garante NUMERIC(15,4) → DecimalType(15,4), sem perda de precisão
df_transacoes = df_transacoes.withColumn("valor", col("valor").cast(DecimalType(15, 4)))

print(f"[INFO] {df_transacoes.count()} transação(ões) extraída(s).")
df_transacoes.printSchema()
df_transacoes.show(5, truncate=False)

[INFO] Extraindo transacoes_financeiras com data_criacao > 2000-01-01 00:00:00
[INFO] 20000 transação(ões) extraída(s).
root
 |-- id_transacao: string (nullable = true)
 |-- conta_origem: string (nullable = true)
 |-- conta_destino: string (nullable = true)
 |-- tipo_movimento: string (nullable = true)
 |-- valor: decimal(15,4) (nullable = true)
 |-- moeda: string (nullable = true)
 |-- data_transacao: timestamp (nullable = true)
 |-- hash_auditoria: string (nullable = true)
 |-- data_criacao: timestamp (nullable = true)

+------------------------------------+------------------+------------------+--------------+----------+-----+-------------------+----------------------------------------------------------------+-------------------+
|id_transacao                        |conta_origem      |conta_destino     |tipo_movimento|valor     |moeda|data_transacao     |hash_auditoria                                                  |data_criacao       |
+------------------------------------+------

In [7]:
# Carga — Bronze MinIO em formato Parquet
# Parquet preserva nativamente DecimalType(15,4), garantindo
# fidelidade dos dados monetários na camada Bronze.
try:
    if df_transacoes.rdd.isEmpty():
        print("[INFO] Nenhuma transação nova. Nenhum arquivo gerado.")
    else:
        (
            df_transacoes
            .coalesce(1)
            .write
            .mode("overwrite")
            .parquet(OUTPUT_PATH)
        )
        print(f"[OK] Arquivo Parquet gravado em: {OUTPUT_PATH}")

    upload_metadata(s3, {"timestamp": TIMESTAMP_INGESTAO, "status": "success"})
    print("FLUXO 3 concluído com sucesso.")

except Exception as e:
    print(f"[ERROR] Ocorreu um erro: {e}")
    upload_metadata(s3, {"timestamp": TIMESTAMP_INGESTAO, "status": "error", "mensagem": str(e)})
    raise

[OK] Arquivo Parquet gravado em: s3a://datalake/bronze/financeiro/ledger/data_ingestao=2026-05-30/transacoes.parquet
[OK] Metadata salvo: s3://datalake/bronze/financeiro/metadata/2026-05-30 15:17:18.json
FLUXO 3 concluído com sucesso.


In [8]:
# Encerramento da SparkSession
spark.stop()
print("SparkSession encerrada.")

SparkSession encerrada.
